In [4]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

MODEL_NAME = "facebook/nllb-200-distilled-600M"

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load tokenizer + model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

def translate(texts, src_lang, tgt_lang, batch_size=32):
    tokenizer.src_lang = src_lang
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    outputs = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_new_tokens=128
            )

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        outputs.extend(decoded)

    return outputs

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [1]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving twitter_test_en.csv to twitter_test_en.csv
Saving twitter_train_en.csv to twitter_train_en.csv
Saving twitter_val_en.csv to twitter_val_en.csv
User uploaded file "twitter_test_en.csv" with length 2471810 bytes
User uploaded file "twitter_train_en.csv" with length 11449779 bytes
User uploaded file "twitter_val_en.csv" with length 2450606 bytes


In [5]:
# Load your YouTube translated file
df = pd.read_csv("twitter_train_en.csv")

# Take small sample (important: don't run on full dataset)
sample_df = df.sample(50, random_state=42)

# Step 1: English → Nepali (back translation)
back_translated = translate(
    sample_df["text_en"].tolist(),
    src_lang="eng_Latn",
    tgt_lang="npi_Deva"
)

sample_df["back_ne"] = back_translated

100%|██████████| 2/2 [00:01<00:00,  1.45it/s]


In [6]:
sample_df[["text", "text_en", "back_ne"]].head(10)

,text,text_en,back_ne
20864,नेपाल में पाँच महिना में अर्ब करोड से अधिक मूल...,Imports of medicines and raw materials worth o...,पाँच महिनामा एक अर्बभन्दा बढीको औषधि र कच्चा म...
5945,आजका ब्रोडसिटका समाचार शीर्षक नेकपा संकट राष्ट...,Today's broadcast news headline is the NECPA c...,"आजको प्रसारण समाचारको शीर्षक नेकपा संकट हो, रा..."
23238,कोभिड का बिरामीलाई अस्पतालसँग समन्वय गरेर मात्...,Referring only a COVID patient in coordination...,अस्पतालसँग समन्वय गरेर मात्र एक जना कोभिड संक्...
16425,नेपालमा कोरोना भाइरस कोभिड को सङ्क्रमण आजको मि...,The coronavirus outbreak in Nepal of COVID-19 ...,नेपालमा कोभिड-१९ को कोरोना भाइरसको प्रकोप अहिल...
20497,कोभिड का संक्रमित बढ्न थालेपछि पोखरा महानगरको ...,"As the number of Covid cases has increased, at...",कोभिड संक्रमितको संख्या बढ्दै जाँदा पोखरा शहरक...
22933,कोभिड बाट हालसम्म मृत्यु हुनेको सङ्ख्या सात जि...,The number of deaths from COVID-19 so far in s...,अहिलेसम्म सात जिल्लामा कोभिड-१९ बाट मृत्यु हुन...
221,कोरोना भाईरस कोभिड का विरामीहरु ओसार्न लगाएर स...,The Swiss Foreign Ministry has not paid its ow...,स्विस विदेश मन्त्रालयले स्विस कम्पनीलाई लाखौं ...
5345,कोभिड संक्रमण पुष्टि भएका गुल्मीको मदाने गाउँप...,The Crimson Hospital has said that the news of...,क्रिमसन अस्पतालले उनको मृत्युको खबर गलत भएको ब...
20715,सार्कका शिक्षामन्त्रीबीच कोभिड पछिको शिक्षाबार...,A review of post-COVID education between the S...,सर्क शिक्षा मन्त्रीहरूबीचको पोस्ट-कोविड शिक्षा...
6760,राष्ट्रपति विद्यादेवी भण्डारीले कोरोना भाइरस क...,President Vidya Devi Bhandari has appealed to ...,राष्ट्रपति विद्यादेवी भण्डारीले कोरोना भाइरसको...


In [ ]:
sample_df.to_csv("twitter_train_en_back_translated.csv", index=False)
print("Back-translated data saved to twitter_train_en_back_translated.csv")

Back-translated data saved to youtube_train_en_back_translated.csv
